<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/THRISHULADHARA_KS_NNM24CS272_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/Smart_College_Campus_Assistant_CrewAI_Nexus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# THRISHULADHARA KS — NNM24CS272 — Assignment 2
**Student: THRISHULADHARA KS (NNM24CS272) | NITTE Phase 1 AI Cohort**

# Smart College Campus Assistant — A Multi-Agent AI System (CrewAI + Nexus API)
**NITTE Phase 1 AI Cohort · Assignment 2 · Due 16 Oct 2026**

## Problem statement
College event coordination (fests, guest visits, announcements) is scattered across chats, calls and spreadsheets. A single chatbot gives generic answers with no division of responsibility, no validated schedules/budgets, and no grounding in real-world data (e.g. weather).

## Objectives
1. Accept natural-language campus event requests.
2. Split work across 4 specialised agents (planning, logistics/budget, weather/risk, communications).
3. Ground outputs with 8 custom tools (deterministic Python where possible, real Open-Meteo API data).
4. Use the **Nexus API for every LLM call** — agents and LLM-powered tools alike.
5. Handle incomplete requests and API failures gracefully, never fabricating facts.

## Proposed solution
A sequential CrewAI crew: Coordinator → Logistics/Budget → Weather/Risk → Communications. Tasks share context (`context=[...]`), tools validate/calculate/fetch, and a formatter merges everything into one coherent response with assumptions and open questions clearly marked.

## Architecture overview
```
User Request -> validate -> Agent1 Coordinator (ReqAnalyzer, ScheduleGen)
  -> Agent2 Logistics/Budget (BudgetCalc, ResourcePlanner) [ctx: Task1]
  -> Agent3 Weather/Risk (WeatherTool live API, RiskTool) [ctx: Task1+2]
  -> Agent4 Communications (AnnouncementTool via Nexus, Formatter) [ctx: Task1-3]
  -> One Combined Final Response (+ debug log, tool-call log)
Shared: Nexus LLM, 8 tools, Open-Meteo, error handling, tests.
```

## Features
- 4 specialised agents, 4 chained tasks, 8 working tools (all demoed + crew-invoked)
- Real weather via Open-Meteo (no key needed), deterministic schedule/budget/risk logic
- Nexus-only LLM usage; secrets via Colab Secrets; no key in code/logs
- 3 functional tests + 2 failure tests; debug-log template; honest observations

## Technology stack
| Component | Choice |
|---|---|
| Agent framework | CrewAI (`crewai[tools]`) — only framework used |
| LLM provider | Nexus API (OpenAI-compatible), model from secret `NEXUSAI_MODEL_ID` (default `gpt-4.1-nano`) |
| External API | Open-Meteo Forecast API (keyless) |
| HTTP | `requests` (timeout + validation) |
| Validation/parsing | `pydantic`, `re`, `json`, `datetime` |
| Runtime | Google Colab notebook (.ipynb) |

## Setup instructions
1. Open this notebook in Google Colab.
2. Add 3 secrets (Colab 🔑 Secrets panel): `NEXUSAI_API_KEY`, `NEXUSAI_BASE_URL`, `NEXUSAI_MODEL_ID`.
3. Run **Runtime → Run all**. Start at Section 2 (installs), then execute top-to-bottom.
4. Test your own request in Section 9 (`user_request = ...`), then re-run Sections 10-12 cells.


## SECTION 2 — Install Dependencies
What each package does:
- `crewai[tools]` — agents, tasks, crews, processes + `BaseTool` interface (the only agent framework).
- `requests` — HTTP client for Open-Meteo and Nexus REST checks.
- `openai` — OpenAI-compatible client used to talk to the **Nexus** endpoint (same pattern as the course Nexus demo).
- `pydantic` — argument schemas + validation for custom tools (usually comes with crewai; listed for clarity).

We install minimal, version-flexible packages so Colab resolves a compatible set.

> ⚠️ **About `ERROR: pip's dependency resolver' lines:** these are **warnings, not failures**. Colab pre-installs packages (`google-colab`, `tweepy`, `bigframes`, `google-adk`) with pinned versions, and CrewAI pulls newer ones. The install still succeeds and the notebook runs fine. If Colab asks to **restart the runtime** after install, do so, then Run all again.


In [2]:
%pip install -q "crewai[tools]" requests openai pydantic


## SECTION 3 — Import Libraries
- CrewAI: `LLM, Agent, Task, Crew, Process` + `BaseTool` (with fallback import for older installs).
- Tools/HTTP: `requests`, `openai` (Nexus-compatible client), `pydantic` schemas.
- Support: `os/re/json/datetime/textwrap/traceback` for validation, parsing, dates, error handling.
- Secrets: Google Colab `userdata` (guarded so the notebook still imports outside Colab).


In [3]:
# --- CrewAI core ---
from crewai import LLM, Agent, Task, Crew, Process
try:
    from crewai.tools import BaseTool          # current supported path
except ImportError:                             # fallback for older installs
    from crewai_tools import BaseTool
# --- Tool schemas / HTTP / utils ---
from pydantic import BaseModel, Field
from typing import Optional
import requests, openai
import os, re, json, textwrap, traceback
from datetime import datetime, timedelta
# --- Secrets (Colab guarded) ---
try:
    from google.colab import userdata
except ImportError:
    userdata = None
print("Imports OK | CrewAI:", __import__("crewai").__version__)


Imports OK | CrewAI: 1.15.28


## SECTION 4 — Nexus API Configuration (ALL LLM calls use Nexus)
**Official pattern reused from the course Nexus demo** (`Text_to_Text.ipynb` in this repo): Nexus is OpenAI-compatible: base URL comes from secret `NEXUSAI_BASE_URL` (default `https://nexusapi.navigatelabs.ai`) and model from `NEXUSAI_MODEL_ID` (default `gpt-4.1-nano`), called via an OpenAI-compatible client. No other provider is used anywhere, and there is **no silent fallback**.

| Setting | Value | Source |
|---|---|---|
| Base URL | `NEXUSAI_BASE_URL` secret (default `https://nexusapi.navigatelabs.ai`) | course Nexus demo notebook |
| Model | `NEXUSAI_MODEL_ID` secret (default `gpt-4.1-nano`) | course Nexus demo notebook |
| Auth | Bearer API key from Colab Secrets (`NEXUSAI_API_KEY`) | never hardcoded/printed |
| CrewAI wiring | `crewai.LLM(model=..., base_url=..., api_key=..., temperature=...)` | verified pattern |

The cell below: reads the key securely (length-only confirmation), builds one shared `nexus_llm` for all agents, exposes `nexus_client()` for LLM-powered tools, and runs a minimal test message. Missing/invalid credentials and network errors are caught with clear messages and a `NEXUSAI_OK` flag.


In [4]:
# ===== Nexus configuration — EDIT NOTHING secret-related here =====
# Secrets (Colab 🔑 Secrets panel) — exact names:
#   NEXUSAI_API_KEY   = Nexus Bearer API key
#   NEXUSAI_BASE_URL  = Nexus OpenAI-compatible endpoint
#   NEXUSAI_MODEL_ID  = Nexus model ID
def _read_secret(name):
    if userdata is not None:
        try:
            v = userdata.get(name)
            if v: return v
        except Exception: pass
    return os.environ.get(name)
NEXUSAI_API_KEY = _read_secret("NEXUSAI_API_KEY")
NEXUSAI_BASE_URL = _read_secret("NEXUSAI_BASE_URL") or "https://nexusapi.navigatelabs.ai"
NEXUSAI_MODEL_ID = _read_secret("NEXUSAI_MODEL_ID") or "gpt-4.1-nano"
print(f"Base URL: {NEXUSAI_BASE_URL} | Model: {NEXUSAI_MODEL_ID} "
      f"(from secrets; defaults shown if secret not set).")
if not NEXUSAI_API_KEY:
    print("ERROR: Nexus key missing. Add Colab Secret 'NEXUSAI_API_KEY', then re-run.")
    NEXUSAI_OK, nexus_llm = False, None
else:
    print(f"Nexus key found (length {len(NEXUSAI_API_KEY)}). Value never printed/logged.")
    try:
        nexus_llm = LLM(model=NEXUSAI_MODEL_ID, base_url=NEXUSAI_BASE_URL, api_key=NEXUSAI_API_KEY, temperature=0.7)
        def nexus_client():  # single Nexus gateway for ALL custom LLM-powered tools
            return openai.OpenAI(api_key=NEXUSAI_API_KEY, base_url=NEXUSAI_BASE_URL)
        # Minimal verification request (no fallback provider on any failure)
        rsp = nexus_client().chat.completions.create(model=NEXUSAI_MODEL_ID, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly: NEXUS-OK"}])
        print("Nexus test reply:", rsp.choices[0].message.content.strip())
        NEXUSAI_OK = True
    except Exception as e:
        print("ERROR: Nexus verification failed:", type(e).__name__, "-", str(e)[:300])
        print("Fix the key/network and re-run. No other LLM provider will be used.")
        NEXUSAI_OK, nexus_llm = False, None


Base URL: https://nexusapi.navigatelabs.ai | Model: gpt-4.1-nano (from secrets; defaults shown if secret not set).
Nexus key found (length 25). Value never printed/logged.
Nexus test reply: NEXUS-OK


## SECTION 5 — External API Research and Testing (Open-Meteo, keyless)
**Why Open-Meteo?** Free, no API key, CORS-friendly, documented at https://open-meteo.com. Perfect for a student notebook.
**Endpoint:** `https://api.open-meteo.com/v1/forecast`
**Key params:** `latitude`, `longitude` (required floats), `current=` (e.g. `temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m`), `daily=` (e.g. `temperature_2m_max,temperature_2m_min,precipitation_probability_max,weathercode`), `timezone=auto`, `forecast_days=1..16` (horizon limit: **max 16 days** — we never claim forecasts beyond that).
**How weather affects planning:** rain/high wind → move outdoor items indoors, add tents/matting; heat (>35°C) → water, shade, reschedule exertion; lightning code → pause outdoor stage. Below we make a **real live request** for NITTE Karkala (13.2146°N, 74.9967°E), validate status + JSON structure, show a small summary, and handle timeouts/bad JSON without fabricating data.


In [5]:
# Independent live test of Open-Meteo (runs BEFORE any tool integration)
WMO = {0:"Clear sky",1:"Mainly clear",2:"Partly cloudy",3:"Overcast",45:"Fog",48:"Icing fog",51:"Light drizzle",53:"Drizzle",55:"Dense drizzle",56:"Freezing drizzle",57:"Freezing drizzle",61:"Slight rain",63:"Rain",65:"Heavy rain",66:"Freezing rain",67:"Freezing rain",71:"Slight snow",73:"Snow",75:"Heavy snow",77:"Snow grains",80:"Slight showers",81:"Showers",82:"Violent showers",85:"Snow showers",86:"Snow showers",95:"Thunderstorm",96:"Storm+hail",99:"Storm+hail"}
NITTE_LAT, NITTE_LON = 13.2146, 74.9967  # NITTE, Karkala, Karnataka — change if your venue differs
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": NITTE_LAT, "longitude": NITTE_LON,
  "current": "temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m",
  "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max,weathercode",
  "timezone": "auto", "forecast_days": 3}
try:
    r = requests.get(url, params=params, timeout=15)
    print("HTTP status:", r.status_code)
    r.raise_for_status()
    data = r.json()  # raises on invalid JSON
    assert "current" in data and "daily" in data, "Unexpected response structure"
    cur, daily = data["current"], data["daily"]
    print(f"Location used: lat {data.get('latitude')}, lon {data.get('longitude')} | tz {data.get('timezone')} | date {cur.get('time')}")
    print(f"Now: {cur.get('temperature_2m')}{data.get('current_units',{}).get('temperature_2m','')} | feels {cur.get('apparent_temperature')} | RH {cur.get('relative_humidity_2m')}% | wind {cur.get('wind_speed_10m')} km/h | {WMO.get(cur.get('weather_code'),cur.get('weather_code'))}")
    for i, d in enumerate(daily["time"]):
        print(f"{d}: {WMO.get(daily['weathercode'][i], daily['weathercode'][i])} | {daily['temperature_2m_min'][i]}..{daily['temperature_2m_max'][i]}C | rain prob {daily['precipitation_probability_max'][i]}%")
    print("\nPlanning read: match outdoor slots against rain-probability and max-temp above; keep indoor backup if prob>=50%.")
    print("Horizon note: Open-Meteo serves max 16 days; dated events beyond that cannot use a real forecast.")
except requests.exceptions.Timeout:
    print("TIMEOUT: weather unavailable (network slow). Planning must continue with contingency, no fabricated forecast.")
except Exception as e:
    print("Weather test failed:", type(e).__name__, "-", str(e)[:400], "\nNo weather data invented.")


HTTP status: 200
Location used: lat 13.181019, lon 74.98633 | tz Asia/Kolkata | date 2026-10-16T12:15
Now: 29.8°C | feels 36.9 | RH 75% | wind 2.3 km/h | Light drizzle
2026-10-16: Showers | 24.2..31.2C | rain prob 100%
2026-10-17: Thunderstorm | 23.4..30.2C | rain prob 100%
2026-10-18: Thunderstorm | 24.0..31.3C | rain prob 98%

Planning read: match outdoor slots against rain-probability and max-temp above; keep indoor backup if prob>=50%.
Horizon note: Open-Meteo serves max 16 days; dated events beyond that cannot use a real forecast.


## SECTION 6a — Custom Tools 1-4 (deterministic Python)

Each tool uses CrewAI's `BaseTool` interface (`name`, `description`, `args_schema`, `_run`) with input validation, a defined JSON return, and error handling. A shared `TOOL_LOG` list records every invocation as demo evidence.

| # | Tool | Agent | Why deterministic? |
|---|---|---|---|
| 1 | Campus Event Requirements Analyzer | Coordinator | keyword/regex parsing beats LLM guessing; separates facts vs assumptions |
| 2 | Event Schedule Generator | Coordinator | arithmetic timetabling; validates no-overlap + fits-duration |
| 3 | Event Budget Calculator | Logistics | Python math for money; LLM arithmetic is untrustworthy |
| 4 | Campus Resource Planner | Logistics | rule-based checklist (essential vs optional) |


In [6]:
# Shared execution log: every tool call appends here as evidence.
TOOL_LOG = []


def _log(tool, args, ok=True, note=""):
    TOOL_LOG.append({"tool": tool, "args": str(args)[:300],
                     "ok": ok, "note": note})
    return TOOL_LOG[-1]


# ---------- TOOL 1: Campus Event Requirements Analyzer ----------
class ReqArgs(BaseModel):
    user_request: str = Field(
        ..., description="Raw natural-language campus event request.")


class RequirementsAnalyzerTool(BaseTool):
    name: str = "Campus Event Requirements Analyzer"
    description: str = (
        "Validate and structure a campus event request into JSON: "
        "event_type, audience, date/duration/venue hints, missing "
        "information and clarifying questions. Separates user-provided "
        "facts from assumptions.")
    args_schema: type[BaseModel] = ReqArgs

    def _run(self, user_request: str) -> str:
        try:
            if not user_request or not user_request.strip():
                _log(self.name, user_request, False, "empty request")
                return json.dumps({"error": "Empty request."})
            t = user_request.lower()
            # Phrase-level check first: guest visits mention guest +
            # visit/speaker/researcher in any word order.
            if ("guest" in t and ("visit" in t or "speaker" in t
                                  or "researcher" in t)):
                etype = "guest visit"
            else:
                etype = "general campus event"
                keys = ["technical fest", "tech fest", "ai fest", "fest",
                        "hackathon", "coding competition", "workshop",
                        "seminar", "guest lecture", "cultural fest",
                        "sports", "orientation", "farewell"]
                for k in keys:
                    if k in t:
                        etype = k
                        break
            m = re.search(
                r"(\d{2,4})\s*(participants|students|attendees|guests|people)",
                t)
            audience = int(m.group(1)) if m else None
            if any(w in t for w in ["one-day", "one day", "1-day",
                                    "single day"]):
                duration = "one-day"
            elif "half" in t:
                duration = "half-day"
            else:
                duration = None
            venue = "NITTE campus" if "nitte" in t else None
            facts = {"event_type": etype}
            if audience:
                facts["audience_approx"] = audience
            if duration:
                facts["duration_hint"] = duration
            if venue:
                facts["venue_hint"] = venue
            date_hit = re.search(r"\d{1,2}[-/ ]\w+|\b\w+ \d{1,2}\b", t)
            missing = [k for k, v in
                       [("audience size", audience), ("date", date_hit),
                        ("venue", venue), ("duration", duration)] if not v]
            out = {
                "facts_user_provided": facts,
                "assumptions": {
                    "duration_default": None if duration else
                    "one-day (09:00-17:00) assumed for planning",
                    "audience_default": None if audience else
                    "150 assumed for planning"},
                "missing_information": missing or ["none critical"],
                "clarifying_questions":
                    [f"Please confirm: {x}?" for x in (missing or [])][:4]}
            _log(self.name, user_request[:80], True, etype)
            return json.dumps(out, indent=2)
        except Exception as e:
            _log(self.name, "err", False, str(e)[:100])
            return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("Tool 1 defined: Campus Event Requirements Analyzer")


Tool 1 defined: Campus Event Requirements Analyzer


In [7]:
# ---------- TOOL 2: Event Schedule Generator ----------
class SchedArgs(BaseModel):
    duration_hours: float = Field(..., description="Total event hours, e.g. 8.")
    activities: str = Field(..., description="Comma-separated activity names.")
    start_time: str = Field("09:00", description="HH:MM 24h start time.")


class ScheduleGeneratorTool(BaseTool):
    name: str = "Event Schedule Generator"
    description: str = (
        "Build a realistic non-overlapping timetable from the event "
        "duration and activity list. Validates that slots fit and do not "
        "overlap. Returns a structured schedule.")
    args_schema: type[BaseModel] = SchedArgs

    def _run(self, duration_hours: float, activities: str,
             start_time: str = "09:00") -> str:
        try:
            acts = [a.strip() for a in activities.split(",") if a.strip()]
            if not acts:
                return json.dumps({"error": "No activities given."})
            if not (1 <= duration_hours <= 16):
                return json.dumps(
                    {"error": "duration_hours must be between 1 and 16."})
            base = datetime.strptime(start_time, "%H:%M")
            slot = (duration_hours * 60) / len(acts)
            if slot < 15:
                return json.dumps(
                    {"error": f"Too many activities for {duration_hours}h "
                              f"(slot {slot:.0f} min < 15 min minimum)."})
            sched, cur = [], base
            for a in acts:
                end = cur + timedelta(minutes=slot)
                sched.append({"activity": a,
                              "start": cur.strftime("%H:%M"),
                              "end": end.strftime("%H:%M")})
                cur = end
            _log(self.name, f"{duration_hours}h/{len(acts)} acts",
                 True, "no-overlap, fits duration")
            return json.dumps(
                {"start": start_time, "total_hours": duration_hours,
                 "slot_minutes": round(slot), "schedule": sched,
                 "validation": "no overlaps; fits within duration"},
                indent=2)
        except ValueError:
            return json.dumps({"error": "start_time must be HH:MM."})
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("Tool 2 defined: Event Schedule Generator")


Tool 2 defined: Event Schedule Generator


In [8]:
# ---------- TOOL 3: Event Budget Calculator ----------
class BudgetArgs(BaseModel):
    participants: int = Field(..., description="Expected headcount.")
    days: int = Field(1, description="Number of event days.")
    food_per_person_per_day: float = Field(
        250.0, description="INR per person per day for food.")
    equipment_lump: float = Field(
        20000.0, description="INR lump sum for stage/equipment.")
    transport_lump: float = Field(
        10000.0, description="INR lump sum for transport.")
    printing_lump: float = Field(
        5000.0, description="INR lump sum for printing/certificates.")
    accommodation_per_person: float = Field(
        0.0, description="INR per outstation guest (0 if none).")
    outstation_guests: int = Field(0, description="Outstation guest count.")
    contingency_pct: float = Field(
        10.0, description="Contingency percentage on subtotal.")


class BudgetCalculatorTool(BaseTool):
    name: str = "Event Budget Calculator"
    description: str = (
        "Estimate event costs in INR: food, accommodation, equipment, "
        "transport, printing plus a configurable contingency. All costs "
        "are labelled ESTIMATES, never vendor quotations.")
    args_schema: type[BaseModel] = BudgetArgs

    def _run(self, participants: int, days: int = 1,
             food_per_person_per_day: float = 250.0,
             equipment_lump: float = 20000.0,
             transport_lump: float = 10000.0,
             printing_lump: float = 5000.0,
             accommodation_per_person: float = 0.0,
             outstation_guests: int = 0,
             contingency_pct: float = 10.0) -> str:
        try:
            if participants <= 0 or days <= 0:
                return json.dumps(
                    {"error": "participants and days must be positive."})
            # Reject negative money inputs (catches bad agent args).
            for label, val in [
                    ("food_per_person_per_day", food_per_person_per_day),
                    ("equipment_lump", equipment_lump),
                    ("transport_lump", transport_lump),
                    ("printing_lump", printing_lump),
                    ("accommodation_per_person", accommodation_per_person),
                    ("outstation_guests", outstation_guests)]:
                if val < 0:
                    return json.dumps(
                        {"error": f"{label} must be >= 0, got {val}."})
            # Contingency must be a sane percentage.
            if not (0 <= contingency_pct <= 100):
                return json.dumps(
                    {"error": ("contingency_pct must be between 0 and 100, "
                                f"got {contingency_pct}.")})
            food = participants * days * food_per_person_per_day
            acc = outstation_guests * accommodation_per_person
            sub = food + acc + equipment_lump + transport_lump + printing_lump
            cont = sub * contingency_pct / 100
            out = {
                "currency": "INR",
                "estimate_disclaimer":
                    "ESTIMATES for planning only - not vendor quotations.",
                "breakdown": {
                    "food": round(food, 2),
                    "accommodation": round(acc, 2),
                    "equipment": equipment_lump,
                    "transport": transport_lump,
                    "printing": printing_lump},
                "subtotal": round(sub, 2),
                "contingency_pct": contingency_pct,
                "contingency": round(cont, 2),
                "total_estimate": round(sub + cont, 2)}
            _log(self.name, f"p={participants},d={days}", True,
                 f"total {out['total_estimate']}")
            return json.dumps(out, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("Tool 3 defined: Event Budget Calculator")


Tool 3 defined: Event Budget Calculator


In [9]:
# ---------- TOOL 4: Campus Resource Planner ----------
class ResArgs(BaseModel):
    participants: int = Field(..., description="Expected headcount.")
    event_type: str = Field("general", description="Event type keywords.")
    outdoor: bool = Field(False, description="True if any outdoor activity.")


class ResourcePlannerTool(BaseTool):
    name: str = "Campus Resource Planner"
    description: str = (
        "List essential vs optional resources (seating, registration, "
        "projectors, mics, internet, volunteers, refreshments) and flag "
        "which details still need confirmation.")
    args_schema: type[BaseModel] = ResArgs

    def _run(self, participants: int, event_type: str = "general",
             outdoor: bool = False) -> str:
        try:
            import math
            if participants <= 0:
                return json.dumps(
                    {"error": "participants must be positive."})
            essential = [
                f"Seating for {participants}",
                f"{math.ceil(participants / 25)} volunteers (1 per 25)",
                "Registration desk + attendee lists",
                "Projector + screen",
                "2 wireless mics + PA system",
                "Wi-Fi for venue + backup hotspot",
                f"Drinking water + refreshments x{participants}"]
            low = event_type.lower()
            if ("coding" in low or "hackathon" in low or "ai" in low
                    or "technical" in low):
                essential += ["Computer lab + power strips/extension boards",
                              "Contest platform + display screens"]
            if outdoor:
                essential += ["Tents/shade + ground mats",
                              "Portable sound + generator backup"]
            optional = ["Live streaming setup", "Photography/videography",
                        "Goodie bags", "Outdoor signage/banners"]
            out = {"essential": essential, "optional": optional,
                   "needs_confirmation": [
                       "Exact venue capacity and fire-safety norms",
                       "Lab booking + power load clearance",
                       "Tent/vendor booking if outdoor"]}
            _log(self.name, f"p={participants},{event_type}", True,
                 f"{len(essential)} essential items")
            return json.dumps(out, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("Tool 4 defined: Campus Resource Planner")


Tool 4 defined: Campus Resource Planner


In [10]:
# Demo: execute Tools 1-4 (definition alone does not count).
t1 = RequirementsAnalyzerTool()
t2 = ScheduleGeneratorTool()
t3 = BudgetCalculatorTool()
t4 = ResourcePlannerTool()

print("== TOOL 1 demo ==")
print(t1._run("Plan a one-day inter-college AI fest at NITTE for "
              "300 students with coding, workshops, keynote, lunch, "
              "closing ceremony."))
print("\n== TOOL 2 demo ==")
print(t2._run(8.0, "Opening, Coding contest, AI workshop, Keynote, "
                    "Lunch, Project demos, Closing ceremony", "09:00"))
print("\n== TOOL 3 demo ==")
print(t3._run(300, 1, 250.0, 20000.0, 10000.0, 5000.0, 0.0, 0, 10.0))
print("\n== TOOL 4 demo ==")
print(t4._run(300, "technical fest", False))


== TOOL 1 demo ==
{
  "facts_user_provided": {
    "event_type": "ai fest",
    "audience_approx": 300,
    "duration_hint": "one-day",
    "venue_hint": "NITTE campus"
  },
  "assumptions": {
    "duration_default": null,
    "audience_default": null
  },
  "missing_information": [
    "none critical"
  ],
  "clarifying_questions": []
}

== TOOL 2 demo ==
{
  "start": "09:00",
  "total_hours": 8.0,
  "slot_minutes": 69,
  "schedule": [
    {
      "activity": "Opening",
      "start": "09:00",
      "end": "10:08"
    },
    {
      "activity": "Coding contest",
      "start": "10:08",
      "end": "11:17"
    },
    {
      "activity": "AI workshop",
      "start": "11:17",
      "end": "12:25"
    },
    {
      "activity": "Keynote",
      "start": "12:25",
      "end": "13:34"
    },
    {
      "activity": "Lunch",
      "start": "13:34",
      "end": "14:42"
    },
    {
      "activity": "Project demos",
      "start": "14:42",
      "end": "15:51"
    },
    {
      "activity"

## SECTION 6b — Custom Tools 5-8 (live weather, Nexus announcement, risk rules, formatter)

- **Tool 5 (Weather):** performs a **real** Open-Meteo HTTP request (15 s timeout, status + JSON validation). On failure it returns an `unavailable` message — never invented weather.
- **Tool 6 (Announcement):** the **LLM-powered** custom tool. It calls **Nexus only** via `nexus_client()` with strict `[TBD]` placeholder rules.
- **Tool 7 (Risk):** transparent Python severity rules + mitigation + human-approval flags.
- **Tool 8 (Formatter):** deterministic Python merge of all sections; preserves uncertainty, never claims bookings were completed.


In [11]:
# ---------- TOOL 5: Weather Information (REAL external API) ----------
class WxArgs(BaseModel):
    latitude: float = Field(..., description="Venue latitude, -90 to 90.")
    longitude: float = Field(..., description="Venue longitude, -180 to 180.")
    forecast_days: int = Field(3, description="1-16 days (API horizon).")


class WeatherInfoTool(BaseTool):
    name: str = "Weather Information API Tool"
    description: str = (
        "Fetch a REAL forecast from the Open-Meteo API for the given "
        "latitude/longitude. Returns actual data or an unavailable message; "
        "never invents weather.")
    args_schema: type[BaseModel] = WxArgs

    def _run(self, latitude: float, longitude: float,
             forecast_days: int = 3) -> str:
        try:
            if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
                return json.dumps({"error": "Invalid latitude/longitude."})
            forecast_days = max(1, min(16, int(forecast_days)))
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": latitude, "longitude": longitude,
                        "current": "temperature_2m,relative_humidity_2m,"
                                   "apparent_temperature,weather_code,"
                                   "wind_speed_10m",
                        "daily": "temperature_2m_max,temperature_2m_min,"
                                 "precipitation_probability_max,weathercode",
                        "timezone": "auto", "forecast_days": forecast_days},
                timeout=15)
            if r.status_code != 200:
                _log(self.name, f"{latitude},{longitude}", False,
                     f"HTTP {r.status_code}")
                return json.dumps(
                    {"status": "unavailable",
                     "reason": f"HTTP {r.status_code}",
                     "guidance": "Use indoor backup; no forecast invented."})
            d = r.json()  # raises on invalid JSON
            cur, dal = d["current"], d["daily"]
            out = {
                "status": "live", "source": "Open-Meteo (real API data)",
                "location": {"lat_requested": [latitude, longitude],
                             "lat_resolved": d.get("latitude"),
                             "lon_resolved": d.get("longitude"),
                             "tz": d.get("timezone")},
                "now": {"time": cur.get("time"),
                        "temp_c": cur.get("temperature_2m"),
                        "feels_c": cur.get("apparent_temperature"),
                        "rh_pct": cur.get("relative_humidity_2m"),
                        "wind_kmh": cur.get("wind_speed_10m"),
                        "wmo_code": cur.get("weather_code")},
                "daily": [{"date": dal["time"][i],
                           "tmin": dal["temperature_2m_min"][i],
                           "tmax": dal["temperature_2m_max"][i],
                           "rain_prob": dal["precipitation_probability_max"][i],
                           "wmo": dal["weathercode"][i]}
                          for i in range(len(dal["time"]))],
                "planning_note": ("Outdoor items need cover if any "
                                  "rain_prob >= 50 or tmax >= 36.")}
            _log(self.name, f"{latitude},{longitude}", True, "live data")
            return json.dumps(out, indent=2)
        except requests.exceptions.Timeout:
            _log(self.name, "timeout", False, "timeout")
            return json.dumps(
                {"status": "unavailable", "reason": "timeout",
                 "guidance": "Proceed with indoor contingency."})
        except Exception as e:
            _log(self.name, "err", False, str(e)[:100])
            return json.dumps(
                {"status": "unavailable",
                 "reason": f"{type(e).__name__}: {str(e)[:200]}"})


print("Tool 5 defined: Weather Information API Tool")


Tool 5 defined: Weather Information API Tool


In [12]:
# ---------- TOOL 6: Campus Announcement Generator (LLM via NEXUS ONLY) ----------
class AnnArgs(BaseModel):
    event_title: str = Field(..., description="Event title.")
    event_details: str = Field(
        ..., description="Confirmed facts plus placeholder context.")
    audience: str = Field("students", description="students|faculty|guests|all.")


class AnnouncementTool(BaseTool):
    name: str = "Campus Announcement Generator"
    description: str = (
        "Generate a title + body announcement from ONLY the provided facts "
        "using the Nexus LLM. Unknown date/venue/link/contact becomes "
        "'[TBD]'. Never invents dates, links or contacts.")
    args_schema: type[BaseModel] = AnnArgs

    def _run(self, event_title: str, event_details: str,
             audience: str = "students") -> str:
        try:
            if not event_title.strip() or not event_details.strip():
                return json.dumps(
                    {"error": "event_title and event_details are required."})
            if not NEXUSAI_API_KEY:
                return json.dumps(
                    {"error": "Nexus key missing; announcement not "
                              "generated. No fallback provider used."})
            sys = ("You draft campus announcements. RULES: use ONLY facts "
                   "in event_details; render anything unknown as '[TBD]'. "
                   "Never invent registrations, bookings, times or contacts. "
                   "Keep 120-180 words, friendly, ending with a "
                   "'Pending confirmations' line.")
            rsp = nexus_client().chat.completions.create(
                model=NEXUSAI_MODEL_ID, temperature=0.5, max_tokens=400,
                messages=[
                    {"role": "system", "content": sys},
                    {"role": "user",
                     "content": f"Title: {event_title}\nAudience: {audience}\n"
                                f"Facts:\n{event_details}"}])
            body = rsp.choices[0].message.content.strip()
            _log(self.name, event_title, True, "via Nexus LLM")
            return json.dumps({"title": event_title, "audience": audience,
                               "body": body,
                               "llm": "Nexus/" + NEXUSAI_MODEL_ID}, indent=2)
        except Exception as e:
            _log(self.name, event_title[:40], False, str(e)[:100])
            return json.dumps(
                {"error": f"Nexus announcement failed: {type(e).__name__}: "
                          f"{str(e)[:300]}"})


print("Tool 6 defined: Campus Announcement Generator (Nexus LLM)")


Tool 6 defined: Campus Announcement Generator (Nexus LLM)


In [13]:
# ---------- TOOL 7: Event Risk Assessment (rule-based) ----------
class RiskArgs(BaseModel):
    event_type: str = Field("general", description="Event type keywords.")
    outdoor: bool = Field(False, description="True if any outdoor activity.")
    participants: int = Field(100, description="Expected headcount.")
    weather_summary: str = Field(
        "unavailable", description="Weather tool output or 'unavailable'.")


class RiskAssessmentTool(BaseTool):
    name: str = "Event Risk Assessment Tool"
    description: str = (
        "Score risks (weather, crowding, tech failure, delays, missing "
        "resources, guest travel) with transparent fixed rules; suggest "
        "mitigations and flag where human approval is required.")
    args_schema: type[BaseModel] = RiskArgs

    def _run(self, event_type: str = "general", outdoor: bool = False,
             participants: int = 100,
             weather_summary: str = "unavailable") -> str:
        try:
            risks = []

            def add(name, sev, mit, human):
                risks.append({"risk": name, "severity": sev,
                              "mitigation": mit, "human_approval": human})

            w = weather_summary.lower()
            if outdoor and ("unavailable" in w or "rain" in w
                            or "storm" in w or "thunder" in w):
                add("Bad weather disrupts outdoor sessions",
                    "HIGH" if outdoor else "MEDIUM",
                    "Keep an indoor backup hall; tents/mats on standby; "
                    "go/no-go call 24h and 6h before.", True)
            if participants > 400:
                sev = "HIGH"
            elif participants > 150:
                sev = "MEDIUM"
            else:
                sev = "LOW"
            add("Overcrowding / crowd build-up", sev,
                "Entry time-bands, hall-wise caps, 1 volunteer per 25 "
                "attendees, first-aid desk.", participants > 250)
            add("Projector / mic / network failure", "MEDIUM",
                "Spare projector + mic, offline contest mirror, "
                "printed schedules.", False)
            add("Session delays cascade", "MEDIUM",
                "10-minute buffers, dedicated timekeeper, shorten Q&A "
                "if running >15 min late.", False)
            add("Missing resources / rooms", "MEDIUM",
                "48-hour checklist sign-off by facilities; labelled "
                "kit boxes per venue.", True)
            if "guest" in event_type.lower() or "visit" in event_type.lower():
                add("Guest travel disruption", "MEDIUM",
                    "Live ETA tracking, backup cab on standby, "
                    "flexible slot order.", True)
            _log(self.name, event_type, True, f"{len(risks)} risks scored")
            return json.dumps(
                {"risks": risks,
                 "rule_note": "Severity from fixed rules on headcount, "
                              "outdoor flag and weather keywords - "
                              "not LLM guessing."}, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("Tool 7 defined: Event Risk Assessment Tool")


Tool 7 defined: Event Risk Assessment Tool


In [14]:
# ---------- TOOL 8: Final Response Formatter (deterministic) ----------
class FmtArgs(BaseModel):
    title: str = Field(..., description="Final response title.")
    sections_json: str = Field(
        ..., description="JSON object mapping section name to text.")


class FinalFormatterTool(BaseTool):
    name: str = "Final Response Formatter"
    description: str = (
        "Deterministically merge schedule, resources, budget, weather, "
        "risks, announcements and next actions into one readable response. "
        "Preserves uncertainty; never claims bookings were completed.")
    args_schema: type[BaseModel] = FmtArgs

    def _run(self, title: str, sections_json: str) -> str:
        try:
            secs = json.loads(sections_json)
            order = ["Event Plan", "Schedule", "Resources",
                     "Budget (estimates)", "Weather (live or unavailable)",
                     "Risks & Contingency", "Announcement",
                     "Assumptions & Missing Info", "Next Actions"]
            parts = [
                f"# {title}",
                "_Auto-assembled from agent outputs. Cost figures are "
                "planning estimates; nothing is booked, purchased or "
                "announced unless an action was actually performed._"]
            for k in order:
                if k in secs:
                    parts += [f"\n## {k}\n{secs[k]}"]
            for k, v in secs.items():
                if k not in order:
                    parts += [f"\n## {k}\n{v}"]
            _log(self.name, title, True, f"{len(secs)} sections merged")
            return "\n".join(parts)
        except Exception as e:
            return json.dumps(
                {"error": f"Formatter needs valid sections_json: {e}"})


print("Tool 8 defined: Final Response Formatter")


Tool 8 defined: Final Response Formatter


In [15]:
# Demo: execute Tools 5-8 (all eight tools must be shown running).
t5 = WeatherInfoTool()
t6 = AnnouncementTool()
t7 = RiskAssessmentTool()
t8 = FinalFormatterTool()

print("== TOOL 5 demo (LIVE Open-Meteo) ==")
wx_out = t5._run(13.2146, 74.9967, 2)
print(wx_out[:1200])

print("\n== TOOL 6 demo (Nexus LLM announcement) ==")
if NEXUSAI_OK:
    print(t6._run(
        "NITTE AI Tech Fest",
        "One-day fest at NITTE campus for ~300 students. Activities: "
        "coding contest, AI workshop, keynote, lunch, closing ceremony. "
        "Date / venue / time: [TBD by organisers].",
        "students")[:1200])
else:
    print("SKIPPED: Nexus key missing - no fallback provider used.")

print("\n== TOOL 7 demo ==")
print(t7._run("technical fest", False, 300, wx_out[:500]))

print("\n== TOOL 8 demo ==")
print(t8._run("Demo Response", json.dumps(
    {"Schedule": "09:00 Opening ...",
     "Budget (estimates)": "INR ~1,10,000 (estimate)",
     "Next Actions": "Confirm date + venue"}))[:800])


== TOOL 5 demo (LIVE Open-Meteo) ==
{
  "status": "live",
  "source": "Open-Meteo (real API data)",
  "location": {
    "lat_requested": [
      13.2146,
      74.9967
    ],
    "lat_resolved": 13.181019,
    "lon_resolved": 74.98633,
    "tz": "Asia/Kolkata"
  },
  "now": {
    "time": "2026-10-16T12:15",
    "temp_c": 29.8,
    "feels_c": 36.9,
    "rh_pct": 75,
    "wind_kmh": 2.3,
    "wmo_code": 51
  },
  "daily": [
    {
      "date": "2026-10-16",
      "tmin": 24.2,
      "tmax": 31.2,
      "rain_prob": 100,
      "wmo": 81
    },
    {
      "date": "2026-10-17",
      "tmin": 23.4,
      "tmax": 30.2,
      "rain_prob": 100,
      "wmo": 95
    }
  ],
  "planning_note": "Outdoor items need cover if any rain_prob >= 50 or tmax >= 36."
}

== TOOL 6 demo (Nexus LLM announcement) ==
{
  "title": "NITTE AI Tech Fest",
  "audience": "students",
  "body": "Join us for the exciting NITTE AI Tech Fest, a one-day event designed for students passionate about technology and artificial 

## SECTION 7 — Multi-Agent Architecture (4 specialised agents)

Each agent has a distinct role, goal, backstory, toolset and the **shared Nexus LLM**. Why this split? Planning needs structuring skill, logistics needs numbers, risk needs outside data, communications needs writing — one agent doing all four mixes concerns and invents facts. The cell below builds the four agents; it refuses to fall back to any other provider if Nexus is unconfigured.


In [16]:
# Instantiate tool objects once and share them with agents.
req_tool = RequirementsAnalyzerTool()
sched_tool = ScheduleGeneratorTool()
budget_tool = BudgetCalculatorTool()
resource_tool = ResourcePlannerTool()
weather_tool = WeatherInfoTool()
announce_tool = AnnouncementTool()
risk_tool = RiskAssessmentTool()
format_tool = FinalFormatterTool()

ALL_TOOLS = {"requirements": req_tool, "schedule": sched_tool,
             "budget": budget_tool, "resources": resource_tool,
             "weather": weather_tool, "announce": announce_tool,
             "risk": risk_tool, "formatter": format_tool}


def build_agents():
    """Create the 4 specialised agents. Nexus LLM only - no fallback."""
    if not NEXUSAI_OK or nexus_llm is None:
        raise RuntimeError(
            "Nexus LLM not configured (see Section 4). Refusing to use "
            "any other provider. Fix NEXUSAI_API_KEY and re-run.")
    coordinator = Agent(
        role="Campus Event Planning Coordinator",
        goal=("Understand the user's requirements, flag missing info, and "
              "establish a clear, structured event plan."),
        backstory=("An experienced campus event coordinator who specialises "
                   "in inter-college festivals, student activities, "
                   "workshops and seminars. Precise about facts vs "
                   "assumptions; never invents dates or venues."),
        tools=[req_tool, sched_tool], llm=nexus_llm, verbose=True)
    logistics = Agent(
        role="Campus Logistics and Budget Manager",
        goal=("Turn the event plan into a resourced, costed logistics "
              "plan with validated numbers."),
        backstory=("An experienced logistics planner managing campus "
                   "facilities, equipment, volunteers, food, transport and "
                   "budgets. Uses Python calculators, not mental math."),
        tools=[budget_tool, resource_tool], llm=nexus_llm, verbose=True)
    risk_analyst = Agent(
        role="Event Risk and Contingency Analyst",
        goal=("Ground the plan in real weather data and score risks with "
              "practical backup options."),
        backstory=("A risk-management specialist covering environmental "
                   "conditions, crowd safety, technical risks and travel "
                   "disruption. Reports 'unavailable' honestly when data "
                   "is missing."),
        tools=[weather_tool, risk_tool], llm=nexus_llm, verbose=True)
    communicator = Agent(
        role="Campus Communications Specialist",
        goal=("Turn prior agents' outputs into clear announcements and one "
              "coherent final response with open questions marked."),
        backstory=("A professional campus communications coordinator "
                   "writing for students, faculty, guests and external "
                   "participants. Uses [TBD] placeholders; claims no "
                   "booking as done unless performed."),
        tools=[announce_tool, format_tool], llm=nexus_llm, verbose=True)
    return coordinator, logistics, risk_analyst, communicator


try:
    agent1, agent2, agent3, agent4 = build_agents()
    print("4 agents ready:", agent1.role, "|", agent2.role, "|",
          agent3.role, "|", agent4.role)
except RuntimeError as e:
    print("AGENTS NOT BUILT:", e)
    agent1 = agent2 = agent3 = agent4 = None


4 agents ready: Campus Event Planning Coordinator | Campus Logistics and Budget Manager | Event Risk and Contingency Analyst | Campus Communications Specialist


## SECTION 8 — Task Definitions and Chaining

Four tasks (one per agent), executed **sequentially** with explicit `context` so downstream agents reuse — not re-invent — upstream facts. This mirrors the proven pattern in this repo's `03_multi_agent_crewai_(3).ipynb` (`resume_task` → `skills_task(context=[resume_task])` → `review_task(context=[resume_task, skills_task])`), which runs end-to-end and prints a final report:
- Task 1 (Coordinator): analyse request + initial timetable.
- Task 2 (Logistics, context=[Task 1]): budget + resources from Task 1 facts.
- Task 3 (Risk, context=[Task 1, Task 2]): live weather (outdoor only) + scored risks.
- Task 4 (Comms, context=[Task 1-3]): Nexus announcement + formatted final response.

Each description **names the exact tool** to call, so tool invocation genuinely happens during `crew.kickoff()`. No circular dependencies. Like the `03` notebook, the crew is built as `Crew(agents=[...], tasks=[...], process=Process.sequential, verbose=...)` and launched with `crew.kickoff()`.


In [17]:
# Build tasks bound to a specific user request. Context chain is explicit.
def build_tasks(a1, a2, a3, a4, user_request):
    t1 = Task(
        description=(
            f"Analyse this campus request: '''{user_request}'''. "
            "1) Call 'Campus Event Requirements Analyzer' with the full "
            "request. 2) From its JSON, draft the event plan; then call "
            "'Event Schedule Generator' (8h day starting 09:00 unless the "
            "request says otherwise) with the activity list. "
            "List missing info and assumptions explicitly. Do NOT invent "
            "dates, venues or counts."),
        expected_output=("Structured JSON-ish plan: event_type, audience, "
                         "draft timetable, missing info, assumptions."),
        agent=a1)
    t2 = Task(
        description=(
            "Using Task 1's plan (via context): 1) Call 'Event Budget "
            "Calculator' with the headcount (default 150 if Task 1 assumed "
            "it) and 'Campus Resource Planner' with participants/event "
            "type/outdoor flag. 2) Return itemised budget (INR estimates), "
            "essential vs optional resources, and confirmations needed. "
            "Never present estimates as vendor quotes."),
        expected_output=("Budget breakdown + totals, resource checklist "
                         "(essential/optional), confirmations list."),
        agent=a2, context=[t1])
    t3 = Task(
        description=(
            "Using Tasks 1-2 outputs: if the event is outdoor or "
            "weather-sensitive, call 'Weather Information API Tool' with "
            "latitude 13.2146, longitude 74.9967 (NITTE Karkala) unless the "
            "request gives other coordinates. Then ALWAYS call 'Event Risk "
            "Assessment Tool' with event_type/outdoor/participants plus the "
            "weather JSON (or the word 'unavailable'). Return live weather "
            "or an honest unavailable note, scored risks and mitigations."),
        expected_output=("Weather section (live or unavailable), risk table "
                         "with severity + mitigations + human-approval flags."),
        agent=a3, context=[t1, t2])
    t4 = Task(
        description=(
            "Using Tasks 1-3 outputs: 1) Call 'Campus Announcement "
            "Generator' with the confirmed title/facts (unknowns stay as "
            "[TBD]). 2) Call 'Final Response Formatter' to merge Event "
            "Plan, Schedule, Resources, Budget, Weather, Risks, "
            "Announcement, Assumptions & Missing Info, Next Actions. "
            "Preserve uncertainty; claim no booking as completed."),
        expected_output=("One coherent final campus response with all "
                         "sections, placeholders and next actions."),
        agent=a4, context=[t1, t2, t3])
    return [t1, t2, t3, t4]


print("Task builder ready: build_tasks(a1..a4, request) -> [t1..t4].")


Task builder ready: build_tasks(a1..a4, request) -> [t1..t4].


## SECTION 9 — User Input Interface

Edit `user_request` for any campus scenario — the pipeline is not locked to the example. The validator below rejects empty/off-topic input, detects missing essentials (date, venue, headcount) and either lists follow-up questions or proceeds with **explicit assumptions** for a provisional plan, without crashing.


In [18]:
# Change this string for any campus scenario you want to try.
user_request = """
Plan a one-day inter-college AI and technology fest at NITTE.
The event will include coding competitions, AI workshops, project
presentations, a keynote session, and a closing ceremony.
Expect approximately 300 participants.
"""


def validate_request(text):
    """Return dict with status ok|empty|offtopic|incomplete-ok."""
    if not text or not text.strip():
        return {"status": "empty", "message": "Request is empty."}
    t = text.lower()
    campus_keys = ["event", "fest", "competition", "workshop", "seminar",
                   "lecture", "visit", "announce", "schedule", "campus",
                   "college", "student", "faculty", "guest", "ceremony",
                   "keynote", "nitte"]
    if not any(k in t for k in campus_keys):
        return {"status": "offtopic",
                "message": "Not a campus event request. Try e.g. "
                           "'Plan a workshop...'."}
    missing = []
    if not re.search(r"\d{2,4}\s*(participants|students|attendees|guests)",
                     t):
        missing.append("headcount")
    if "nitte" not in t and "venue" not in t and "campus" not in t:
        missing.append("venue")
    if not re.search(r"\d{1,2}[-/ ]\w+|\b\w+ \d{1,2}\b|tomorrow|next week", t):
        missing.append("date")
    return {"status": "ok", "missing": missing,
            "followups": [f"Please confirm: {m}?" for m in missing]}


print("Request under test:")
print(textwrap.fill(user_request.strip(), 90))
print("\nValidation:", json.dumps(validate_request(user_request), indent=2))


Request under test:
Plan a one-day inter-college AI and technology fest at NITTE. The event will include
coding competitions, AI workshops, project presentations, a keynote session, and a closing
ceremony. Expect approximately 300 participants.

Validation: {
  "status": "ok",
  "missing": [],
  "followups": []
}


## SECTION 10 — Workflow Execution (`run_campus_assistant`)

One reusable function does everything: validate input → build agents/tasks → run the crew **sequentially** → collect intermediate outputs → print the tool-call log (with real weather labelled) → return the final response plus a debug bundle. Failures are caught per-stage with diagnostics; secrets never appear in logs. The next cell runs the Section 9 request through it.


In [19]:
# Reusable end-to-end runner. All LLM work stays on Nexus.
# NOTE (Debug Log #3): Colab/Jupyter already runs an event loop, so sync
# crew.kickoff() raises "invoked synchronously from within a running event
# loop". Fix: prefer crew.kickoff_async() via asyncio, with fallbacks.
# (This repo's 03_multi_agent notebook uses `await crew.kickoff_async()`.)
import asyncio as _asyncio
def _crew_run(crew):
    async def _go():
        return await crew.kickoff_async()
    try:
        _asyncio.get_running_loop()
        running = True
    except RuntimeError:
        running = False
    if running:
        # Inside Colab's loop: run kickoff_async in a FRESH thread+loop.
        import concurrent.futures as _cf
        with _cf.ThreadPoolExecutor(max_workers=1) as _ex:
            return _ex.submit(_asyncio.run, _go()).result()
    try:
        return _asyncio.run(_go())
    except RuntimeError:
        return crew.kickoff()


def run_campus_assistant(user_request, verbose=False):
    bundle = {"input": user_request, "stages": {}, "tools_used": [],
              "final": "", "error": None}
    try:
        v = validate_request(user_request)
        bundle["stages"]["validation"] = v
        if v["status"] in ("empty", "offtopic"):
            bundle["final"] = f"Cannot proceed: {v['message']}"
            return bundle
        a1, a2, a3, a4 = build_agents()
        tasks = build_tasks(a1, a2, a3, a4, user_request.strip())
        bundle["stages"]["task_plan"] = [
            f"T{i+1} -> {t.agent.role}: {t.description[:90]}..."
            for i, t in enumerate(tasks)]
        crew = Crew(agents=[a1, a2, a3, a4], tasks=tasks,
                    process=Process.sequential, verbose=verbose)
        result = _crew_run(crew)
        bundle["final"] = str(result)
        # Collect the outputs stored on the executed CrewAI tasks.
        # Each Task keeps its result in .output after kickoff.
        inter = {}
        for i, t in enumerate(tasks):
            key = f"task_{i+1}_{t.agent.role}"
            try:
                o = getattr(t, "output", None)
                if o is None:
                    inter[key] = "(no output)"
                elif hasattr(o, "raw"):
                    inter[key] = str(o.raw)[:1500]
                else:
                    inter[key] = str(o)[:1500]
            except Exception as e:
                inter[key] = f"(output read failed: {type(e).__name__})"
        bundle["stages"]["intermediate"] = inter
        bundle["tools_used"] = [
            {"tool": e["tool"], "ok": e["ok"], "note": e["note"]}
            for e in TOOL_LOG]
        return bundle
    except Exception as e:
        bundle["error"] = f"{type(e).__name__}: {str(e)[:500]}"
        bundle["final"] = ("Workflow failed safely. Partial stages kept. "
                           f"Diagnostic: {bundle['error']}")
        return bundle


print("Runner ready: run_campus_assistant(user_request).")


Runner ready: run_campus_assistant(user_request).


In [20]:
# Execute the Section 9 request (verbose=False keeps output readable).
if NEXUSAI_OK and agent1 is not None:
    TOOL_LOG.clear()
    out = run_campus_assistant(user_request, verbose=False)
    print("ORIGINAL REQUEST:\n", out["input"].strip())
    print("\nTASK PLAN:")
    for line in out["stages"].get("task_plan", []):
        print(" -", line)
    print("\nTOOLS INVOKED:")
    for e in out["tools_used"]:
        print(f" - {e['tool']} (ok={e['ok']}) :: {e['note']}")
    print("\n===== FINAL RESPONSE =====\n")
    print(out["final"])
    if out["error"]:
        print("\nDIAGNOSTIC:", out["error"])
else:
    print("Crew run skipped: configure NEXUSAI_API_KEY first (no fallback).")
    print("You can still review Sections 5-6 tool demos above.")


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Task: Analyse this campus request: '''Plan a one-day inter-college AI and technology fest at NITTE.            │
│  The event will include coding competitions, AI workshops, project                                              │
│  presentations, a keynote session, and a closing ceremony.                                                      │
│  Expect approximately 300 participants.'''. 1) Call 'Campus Event Requirements Analyzer' with the full          │
│  request. 2) From its JSON, draft the event plan; then call 'Event Schedule Generator' (8h day starting 09:00   │
│  unless the request says otherwise) with the activity list. List missing info and assumptions explicitly. Do    │
│  NOT invent dates, venues or counts.                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_event_requirements_analyzer executed with result: {
  "facts_user_provided": {
    "event_type": "fest",
    "audience_approx": 300,
    "duration_hint": "one-day",
    "venue_hint": "NITTE campus"
  },
  "assumptions": {
    "duration_default": null...
Tool event_schedule_generator executed with result: {
  "start": "09:00",
  "total_hours": 8.0,
  "slot_minutes": 96,
  "schedule": [
    {
      "activity": "Coding Competitions",
      "start": "09:00",
      "end": "10:36"
    },
    {
      "activi...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "event_type": "Inter-college AI and Technology Fest",                                                        │
│    "audience": "Approximately 300 participants",                                                                │
│    "draft_timetable": [                                                                                         │
│      {                                                                                                          │
│        "activity": "Coding Competitions",                                                                       │
│        "start_time": "09:00",                                                                                   │
│        "end_time": "10:36"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "AI Workshops",                                                                              │
│        "start_time": "10:36",                                                                                   │
│        "end_time": "12:12"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Project Presentations",                                                                     │
│        "start_time": "12:12",                                                                                   │
│        "end_time": "13:48"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Keynote Session",                                                                           │
│        "start_time": "13:48",                                                                                   │
│        "end_time": "15:24"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Closing Ceremony",                                                                          │
│        "start_time": "15:24",                                                                                   │
│        "end_time": "17:00"                                                                                      │
│      }                                                                                                          │
│    ],                                                  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Logistics and Budget Manager                                                                     │
│                                                                                                                 │
│  Task: Using Task 1's plan (via context): 1) Call 'Event Budget Calculator' with the headcount (default 150 if  │
│  Task 1 assumed it) and 'Campus Resource Planner' with participants/event type/outdoor flag. 2) Return          │
│  itemised budget (INR estimates), essential vs optional resources, and confirmations needed. Never present      │
│  estimates as vendor quotes.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool event_budget_calculator executed with result: {
  "currency": "INR",
  "estimate_disclaimer": "ESTIMATES for planning only - not vendor quotations.",
  "breakdown": {
    "food": 75000.0,
    "accommodation": 0.0,
    "equipment": 20000.0,
    "t...
Tool campus_resource_planner executed with result: {
  "essential": [
    "Seating for 150",
    "6 volunteers (1 per 25)",
    "Registration desk + attendee lists",
    "Projector + screen",
    "2 wireless mics + PA system",
    "Wi-Fi for venue + b...
[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Event Risk and Contingency Analyst                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Weather Information:                                                                                           │
│  The live weather data for NITTE Karkala indicates a temperature of 29.8°C with a feels-like temperature of     │
│  36.9°C, humidity at 75%, and minimal wind at 2.3 km/h. The forecast for today shows a high temperature of      │
│  31.2°C and a 100% probability of rain, suggesting persistent rain throughout the day. Outdoor activities may   │
│  require cover if rain persists or intensifies, particularly if rain probability exceeds 50% or temperature     │
│  reaches critical thresholds.                                                                                   │
│                                                                                                                 │
│  Event Risk Assessment:                                                                                         │
│  | Risk                                | Severity | Mitigation                                                  │
│  | Human Approval Needed |                                                                                      │
│  |-------------------------------------|----------|-----------------------------------------------------------  │
│  -----------------------------------|-----------------------|                                                   │
│  | Overcrowding / crowd build-up    | MEDIUM   | Entry time-bands, hall-wise caps, 1 volunteer per 25           │
│  attendees, first-aid desk.           | Yes                   |                                                 │
│  | Projector / mic / network failure | MEDIUM   | Spare projector + mic, offline contest mirror, printed        │
│  schedules.                        | No                    |                                                    │
│  | Session delays cascade            | MEDIUM   | 10-minute buffers, dedicated timekeeper, shorten Q&A if       │
│  running >15 min late.            | No                    |                                                     │
│  | Missing resources / rooms         | MEDIUM   | 48-hour checklist sign-off by facilities; labelled kit boxes  │
│  per venue.                | Yes                   |                                                            │
│                                                                                                                 │
│  Risks are primarily medium in severity, with specific attention needed for crowd management and resource       │
│  readiness. Due to the high probability of rain, contingency plans such as sheltered outdoor setups or          │
│  alternative indoor arrangements should be prepared to mitigate weather-related disruptions. Human approval is  │
│  required for crowd control measures and resource sign-offs to ensure safety and preparedness.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Task: Using Tasks 1-3 outputs: 1) Call 'Campus Announcement Generator' with the confirmed title/facts          │
│  (unknowns stay as [TBD]). 2) Call 'Final Response Formatter' to merge Event Plan, Schedule, Resources,         │
│  Budget, Weather, Risks, Announcement, Assumptions & Missing Info, Next Actions. Preserve uncertainty; claim    │
│  no booking as completed.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_announcement_generator executed with result: {
  "title": "Inter-college AI and Technology Fest",
  "audience": "Students, faculty, guests, and external participants",
  "body": "Join us for the exciting Inter-college AI and Technology Fest at N...
Tool final_response_formatter executed with result: {"error": "Formatter needs valid sections_json: Expecting ',' delimiter: line 9 column 3 (char 1357)"}...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Campus Announcement and Event Planning Overview for Inter-college AI and Technology Fest**                   │
│                                                                                                                 │
│  **Event Plan:**                                                                                                │
│  The event is planned to be an 8-hour fest starting at 09:00 with activities including coding competitions, AI  │
│  workshops, project presentations, a keynote session, and a closing ceremony. The exact date and venue details  │
│  are still to be confirmed.                                                                                     │
│                                                                                                                 │
│  **Schedule:**                                                                                                  │
│  The preliminary timetable includes activities from 09:00 to 17:00 with specific activities scheduled           │
│  sequentially. Exact timings and activity numbers per activity are yet to be finalized.                         │
│                                                                                                                 │
│  **Resources:**                                                                                                 │
│  Resources needed include seating, volunteers, registration desk, AV equipment, Wi-Fi, refreshments, and        │
│  potentially outdoor signage. Confirmations regarding venue capacity, lab bookings, and outdoor arrangements    │
│  are pending.                                                                                                   │
│                                                                                                                 │
│  **Budget:**                                                                                                    │
│  Estimated total budget is INR 1,32,000 covering food, equipment, transport, printing, and contingency funds.   │
│                                                                                                                 │
│  **Weather:**                                                                                                   │
│  Current weather data indicates a high probability of rain with temperatures around 30°C, which could impact    │
│  outdoor activities. Contingency plans should be in place.                                                      │
│                                                                                                                 │
│  **Risks:**                                                                                                     │
│  Risks include overcrowding, technical failures, session delays, and resource shortages. Mitigation strategies  │
│  involve crowd control, backup equipment, buffer times, and resource checklists.                                │
│                                                                                                                 │
│  **Announcement:**                                                                                              │
│  A campus-wide announcement will be issued to inform all stakeholders about the event details once confirmed.   │
│                                                        

ORIGINAL REQUEST:
 Plan a one-day inter-college AI and technology fest at NITTE.
The event will include coding competitions, AI workshops, project
presentations, a keynote session, and a closing ceremony.
Expect approximately 300 participants.

TASK PLAN:
 - T1 -> Campus Event Planning Coordinator: Analyse this campus request: '''Plan a one-day inter-college AI and technology fest at NIT...
 - T2 -> Campus Logistics and Budget Manager: Using Task 1's plan (via context): 1) Call 'Event Budget Calculator' with the headcount (d...
 - T3 -> Event Risk and Contingency Analyst: Using Tasks 1-2 outputs: if the event is outdoor or weather-sensitive, call 'Weather Infor...
 - T4 -> Campus Communications Specialist: Using Tasks 1-3 outputs: 1) Call 'Campus Announcement Generator' with the confirmed title/...

TOOLS INVOKED:
 - Campus Event Requirements Analyzer (ok=True) :: fest
 - Event Schedule Generator (ok=True) :: no-overlap, fits duration
 - Event Budget Calculator (ok=True) :: total 13200

## SECTION 11 — Testing and Observations (3 scenarios)

Each test below shows: exact input → crew run → tools actually invoked → final response (whatever it really is) → honest observations. **Do not edit the printed outputs after running** — the assignment forbids fabricated logs. If you run without a Nexus key, record that as the observation (crew correctly refuses non-Nexus providers) and rely on the Section 6 tool demos for executed evidence.


In [21]:
# TEST 1: Technical fest (schedule, resources, budget, weather, risks, announcement).
test1 = ("Plan a one-day inter-college AI and technology fest at NITTE "
         "for approximately 300 students. Include coding competitions, "
         "AI workshops, a keynote session, lunch, and a closing ceremony.")
print("TEST 1 INPUT:", test1)
if NEXUSAI_OK:
    TOOL_LOG.clear()
    r1 = run_campus_assistant(test1)
    print("\nTOOLS:", [(e["tool"], e["ok"]) for e in r1["tools_used"]])
    print("\nFINAL:\n", r1["final"][:4000])
    print("\nOBSERVATIONS (fill from actual run): schedule realistic? "
          "budget labelled estimates? weather live or honestly "
          "unavailable? announcement free of invented links?")
else:
    print("NOT RUN: Nexus key missing. Observation: runner refuses to use "
          "any other provider, which is the required safe behaviour.")


TEST 1 INPUT: Plan a one-day inter-college AI and technology fest at NITTE for approximately 300 students. Include coding competitions, AI workshops, a keynote session, lunch, and a closing ceremony.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Task: Analyse this campus request: '''Plan a one-day inter-college AI and technology fest at NITTE for         │
│  approximately 300 students. Include coding competitions, AI workshops, a keynote session, lunch, and a         │
│  closing ceremony.'''. 1) Call 'Campus Event Requirements Analyzer' with the full request. 2) From its JSON,    │
│  draft the event plan; then call 'Event Schedule Generator' (8h day starting 09:00 unless the request says      │
│  otherwise) with the activity list. List missing info and assumptions explicitly. Do NOT invent dates, venues   │
│  or counts.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_event_requirements_analyzer executed with result: {
  "facts_user_provided": {
    "event_type": "fest",
    "audience_approx": 300,
    "duration_hint": "one-day",
    "venue_hint": "NITTE campus"
  },
  "assumptions": {
    "duration_default": null...
Tool event_schedule_generator executed with result: {
  "start": "09:00",
  "total_hours": 8.0,
  "slot_minutes": 96,
  "schedule": [
    {
      "activity": "Coding competitions",
      "start": "09:00",
      "end": "10:36"
    },
    {
      "activi...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "event_type": "inter-college AI and technology fest",                                                        │
│    "audience": "approximately 300 students",                                                                    │
│    "draft_timetable": [                                                                                         │
│      {                                                                                                          │
│        "activity": "Coding competitions",                                                                       │
│        "start_time": "09:00",                                                                                   │
│        "end_time": "10:36"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "AI workshops",                                                                              │
│        "start_time": "10:36",                                                                                   │
│        "end_time": "12:12"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Keynote session",                                                                           │
│        "start_time": "12:12",                                                                                   │
│        "end_time": "13:48"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Lunch",                                                                                     │
│        "start_time": "13:48",                                                                                   │
│        "end_time": "15:24"                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Closing ceremony",                                                                          │
│        "start_time": "15:24",                                                                                   │
│        "end_time": "17:00"                                                                                      │
│      }                                                                                                          │
│    ],                                                  

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Logistics and Budget Manager                                                                     │
│                                                                                                                 │
│  Task: Using Task 1's plan (via context): 1) Call 'Event Budget Calculator' with the headcount (default 150 if  │
│  Task 1 assumed it) and 'Campus Resource Planner' with participants/event type/outdoor flag. 2) Return          │
│  itemised budget (INR estimates), essential vs optional resources, and confirmations needed. Never present      │
│  estimates as vendor quotes.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Event Risk and Contingency Analyst                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Weather:                                                                                                       │
│  The forecast for 16th October 2026 at NITTE Karkala indicates a high probability of rain (100%) with maximum   │
│  temperatures reaching 31.2°C and minimum temperatures around 24.2°C. The current feel-like temperature is      │
│  36.9°C with 75% humidity. Outdoor items will need cover due to the high likelihood of rain and high            │
│  temperatures.                                                                                                  │
│                                                                                                                 │
│  Risk Assessment:                                                                                               │
│  - Bad weather disrupts outdoor sessions: HIGH severity. Mitigation includes keeping an indoor backup hall,     │
│  tents, or mats on standby, and making go/no-go decisions 24 hours and 6 hours before the event, requiring      │
│  human approval.                                                                                                │
│  - Overcrowding / crowd build-up: MEDIUM severity. Mitigation involves entry time-bands, hall-wise caps,        │
│  deploying 1 volunteer per 25 attendees, and setting up a first-aid desk, requiring human approval.             │
│  - Projector / mic / network failure: MEDIUM severity. Mitigation includes having spare projectors and          │
│  microphones, an offline contest mirror, and printed schedules. No human approval needed.                       │
│  - Session delays cascade: MEDIUM severity. Mitigation involves 10-minute buffers, a dedicated timekeeper, and  │
│  shortening Q&A sessions if running more than 15 minutes late. No human approval needed.                        │
│  - Missing resources / rooms: MEDIUM severity. Mitigation includes a 48-hour checklist sign-off by facilities   │
│  and labelled kit boxes per venue, requiring human approval.                                                    │
│                                                                                                                 │
│  Summary:                                                                                                       │
│  Given the weather forecast, indoor backup arrangements are strongly recommended. The event planning team       │
│  should confirm indoor venue availability and prepare tents or mats for outdoor use. Additionally, coordinate   │
│  with volunteers and facilities for resource readiness and crowd management.                                    │
│                                                                                                                 │
│  Please advise on indoor venue confirmation and any further preparations needed.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Task: Using Tasks 1-3 outputs: 1) Call 'Campus Announcement Generator' with the confirmed title/facts          │
│  (unknowns stay as [TBD]). 2) Call 'Final Response Formatter' to merge Event Plan, Schedule, Resources,         │
│  Budget, Weather, Risks, Announcement, Assumptions & Missing Info, Next Actions. Preserve uncertainty; claim    │
│  no booking as completed.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_announcement_generator executed with result: {
  "title": "Inter-College AI and Technology Fest 2026",
  "audience": "students, faculty, guests, external participants",
  "body": "Join us for the Inter-College AI and Technology Fest 2026, an exc...
Tool final_response_formatter executed with result: {"error": "Formatter needs valid sections_json: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)"}...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Inter-College AI and Technology Fest 2026**                                                                  │
│                                                                                                                 │
│  **Audience:** students, faculty, guests, external participants                                                 │
│                                                                                                                 │
│  **Body:**                                                                                                      │
│  Join us for the Inter-College AI and Technology Fest 2026, an exciting event designed to bring together        │
│  students, faculty, guests, and external participants! This engaging day will feature a variety of activities   │
│  including coding competitions, AI workshops, a keynote session, lunch, and a closing ceremony. The event aims  │
│  to inspire and challenge approximately 300 students across various disciplines. Activities are scheduled to    │
│  run sequentially from 09:00 to 17:00, promising a full day of innovation and learning.                         │
│                                                                                                                 │
│  Please note that specific details such as the exact date, venue within the NITTE campus, and information       │
│  about speakers and activities are yet to be confirmed. Stay tuned for updates and prepare to immerse yourself  │
│  in the latest advancements in AI and technology.                                                               │
│                                                                                                                 │
│  We look forward to your participation in this vibrant celebration of innovation!                               │
│                                                                                                                 │
│  **Pending confirmations:**                                                                                     │
│  - Exact date of the event                                                                                      │
│  - Venue details within NITTE campus                                                                            │
│  - Number of coding competition rounds or teams                                                                 │
│  - Workshop topics and trainers                                                                                 │
│  - Keynote speaker details                                                                                      │
│  - Logistics for lunch and refreshments                                                                         │
│  - Event registration process and timings                                                                       │
│  - Facilities for activities (e.g., computer labs, auditoriums)                                                 │
│                                                                                                                 │
│  **Next steps:**                                                                                                │
│  The event planning team should confirm indoor venue availability given the weather forecast indicating a high  │
│  probability of rain. Arrangements for tents or mats fo


TOOLS: [('Campus Event Requirements Analyzer', True), ('Event Schedule Generator', True), ('Event Budget Calculator', True), ('Campus Resource Planner', True), ('Weather Information API Tool', True), ('Event Risk Assessment Tool', True), ('Event Risk Assessment Tool', True), ('Campus Announcement Generator', True)]

FINAL:
 **Inter-College AI and Technology Fest 2026**

**Audience:** students, faculty, guests, external participants

**Body:**  
Join us for the Inter-College AI and Technology Fest 2026, an exciting event designed to bring together students, faculty, guests, and external participants! This engaging day will feature a variety of activities including coding competitions, AI workshops, a keynote session, lunch, and a closing ceremony. The event aims to inspire and challenge approximately 300 students across various disciplines. Activities are scheduled to run sequentially from 09:00 to 17:00, promising a full day of innovation and learning.

Please note that specific detai

In [22]:
# TEST 2: Guest speaker visit (itinerary, logistics, confirmations, risks, guest mail draft).
test2 = ("Organise a campus visit for a guest AI researcher arriving from "
         "Bengaluru. Include a welcome session, keynote lecture, student "
         "interaction, lunch, transport coordination, and a departure plan.")
print("TEST 2 INPUT:", test2)
if NEXUSAI_OK:
    TOOL_LOG.clear()
    r2 = run_campus_assistant(test2)
    print("\nTOOLS:", [(e["tool"], e["ok"]) for e in r2["tools_used"]])
    print("\nFINAL:\n", r2["final"][:4000])
    print("\nCHECKS: itinerary present? costs only where info suffices? "
          "guest name/time/booking NOT invented? confirmations listed?")
else:
    print("NOT RUN: Nexus key missing - safe refusal recorded.")


TEST 2 INPUT: Organise a campus visit for a guest AI researcher arriving from Bengaluru. Include a welcome session, keynote lecture, student interaction, lunch, transport coordination, and a departure plan.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Task: Analyse this campus request: '''Organise a campus visit for a guest AI researcher arriving from          │
│  Bengaluru. Include a welcome session, keynote lecture, student interaction, lunch, transport coordination,     │
│  and a departure plan.'''. 1) Call 'Campus Event Requirements Analyzer' with the full request. 2) From its      │
│  JSON, draft the event plan; then call 'Event Schedule Generator' (8h day starting 09:00 unless the request     │
│  says otherwise) with the activity list. List missing info and assumptions explicitly. Do NOT invent dates,     │
│  venues or counts.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_event_requirements_analyzer executed with result: {
  "facts_user_provided": {
    "event_type": "guest visit"
  },
  "assumptions": {
    "duration_default": "one-day (09:00-17:00) assumed for planning",
    "audience_default": "150 assumed for plan...
Tool event_schedule_generator executed with result: {
  "start": "09:00",
  "total_hours": 8.0,
  "slot_minutes": 80,
  "schedule": [
    {
      "activity": "Welcome Session",
      "start": "09:00",
      "end": "10:20"
    },
    {
      "activity":...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Event Planning Coordinator                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "event_type": "guest visit",                                                                                 │
│    "audience": "150 (assumed for planning)",                                                                    │
│    "draft timetable": [                                                                                         │
│      {                                                                                                          │
│        "activity": "Welcome Session",                                                                           │
│        "start": "09:00",                                                                                        │
│        "end": "10:20"                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Keynote Lecture",                                                                           │
│        "start": "10:20",                                                                                        │
│        "end": "11:40"                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Student Interaction",                                                                       │
│        "start": "11:40",                                                                                        │
│        "end": "13:00"                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Lunch",                                                                                     │
│        "start": "13:00",                                                                                        │
│        "end": "14:20"                                                                                           │
│      },                                                                                                         │
│      {                                                                                                          │
│        "activity": "Transport Coordination",                                                                    │
│        "start": "14:20",                                                                                        │
│        "end": "15:40"                                                                                           │
│      },                                                                                                         │
│      {                                                 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Logistics and Budget Manager                                                                     │
│                                                                                                                 │
│  Task: Using Task 1's plan (via context): 1) Call 'Event Budget Calculator' with the headcount (default 150 if  │
│  Task 1 assumed it) and 'Campus Resource Planner' with participants/event type/outdoor flag. 2) Return          │
│  itemised budget (INR estimates), essential vs optional resources, and confirmations needed. Never present      │
│  estimates as vendor quotes.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Event Risk and Contingency Analyst                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Weather:                                                                                                       │
│  The current weather forecast for NITTE Karkala indicates a temperature of approximately 29.8°C with a high     │
│  likelihood of rain (rain probability 100%) and a maximum temperature forecast of 31.2°C. The weather is humid  │
│  with a relative humidity of 75%. Given the high probability of rain and the temperature reaching close to      │
│  36°C, outdoor activities should be covered if any rain is anticipated, especially during the event hours.      │
│                                                                                                                 │
│  Event Risk Assessment:                                                                                         │
│  - Overcrowding / crowd build-up: Low severity. Mitigation includes entry time-bands, hall-wise caps,           │
│  volunteers, and first-aid desks.                                                                               │
│  - Projector / mic / network failure: Medium severity. Mitigation includes spare equipment, offline backups,    │
│  and printed schedules.                                                                                         │
│  - Session delays cascade: Medium severity. Mitigation includes buffers, a dedicated timekeeper, and shortened  │
│  Q&A sessions if delays occur.                                                                                  │
│  - Missing resources / rooms: Medium severity. Mitigation involves a checklist sign-off and labelled kit        │
│  boxes. Human approval needed.                                                                                  │
│  - Guest travel disruption: Medium severity. Mitigation includes live ETA tracking, backup cabs, and flexible   │
│  slot management. Human approval needed.                                                                        │
│                                                                                                                 │
│  Summary:                                                                                                       │
│  - Weather is live and indicates a high likelihood of rain with warm temperatures.                              │
│  - Risks are mostly medium, with some requiring human approval for resource management.                         │
│                                                                                                                 │
│  Mitigations are in place for most risks, but due to the weather forecast, outdoor activities should be         │
│  planned with contingency covers or considered for indoor alternatives if rain is expected during event hours.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Task: Using Tasks 1-3 outputs: 1) Call 'Campus Announcement Generator' with the confirmed title/facts          │
│  (unknowns stay as [TBD]). 2) Call 'Final Response Formatter' to merge Event Plan, Schedule, Resources,         │
│  Budget, Weather, Risks, Announcement, Assumptions & Missing Info, Next Actions. Preserve uncertainty; claim    │
│  no booking as completed.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool campus_announcement_generator executed with result: {
  "title": "Guest Visit at NITTE Karkala",
  "audience": "150 (assumed for planning)",
  "body": "Join us at NITTE Karkala for an exciting Guest Visit scheduled from 09:00 to 17:00. This event featu...
Tool final_response_formatter executed with result: # Upcoming Guest Visit Event at NITTE Karkala
_Auto-assembled from agent outputs. Cost figures are planning estimates; nothing is booked, purchased or announced unless an action was actually performed...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Campus Communications Specialist                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "Event Plan": "The event is scheduled from 09:00 to 17:00, including a welcome session, keynote lecture,     │
│  student interaction, lunch, transport coordination, and departure planning. The exact date and venue are yet   │
│  to be confirmed. The schedule includes buffer times to handle potential delays.",                              │
│    "Schedule": "The tentative timetable is as follows: 09:00-10:20 Welcome Session, 10:20-11:40 Keynote         │
│  Lecture, 11:40-13:00 Student Interaction, 13:00-14:20 Lunch, 14:20-15:40 Transport Coordination, 15:40-17:00   │
│  Departure Plan. Exact timings and activities may be adjusted based on final confirmations.",                   │
│    "Resources": "Essential resources include seating for 150 attendees, AV equipment, Wi-Fi, volunteers,        │
│  registration desk, and refreshments. Optional resources such as live streaming, photography, and banners are   │
│  considered. Confirmations are needed for venue capacity, fire safety, lab booking, and outdoor vendor          │
│  arrangements.",                                                                                                │
│    "Budget": "The estimated total cost is INR 1,26,500, covering food, equipment, transport, printing, and      │
│  contingency funds.",                                                                                           │
│    "Weather": "Forecast indicates a high likelihood of rain (100%) with temperatures around 29.8°C to 31.2°C,   │
│  and high humidity. Outdoor activities should include contingency plans or be moved indoors.",                  │
│    "Risks": "Potential risks include overcrowding, equipment failure, session delays, missing resources, and    │
│  guest travel disruptions. Medium severity mitigations are in place, including backups, buffers, and human      │
│  approvals.",                                                                                                   │
│    "Announcement": "A campus announcement will be issued once all details are finalized, including venue,       │
│  date, and resource confirmations.",                                                                            │
│    "Assumptions & Missing Info": "Assumptions include a one-day event from 09:00 to 17:00 and 150 attendees.    │
│  Missing details are the exact date, venue, and precise audience size.",                                        │
│    "Next Actions": "Confirm venue capacity, finalize date, secure outdoor vendor bookings if needed, and        │
│  communicate contingency plans for weather. Human approval required for resource adjustments and travel         │
│  arrangements."                                                                                                 │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


TOOLS: [('Campus Event Requirements Analyzer', True), ('Event Schedule Generator', True), ('Event Budget Calculator', True), ('Campus Resource Planner', True), ('Event Risk Assessment Tool', True), ('Weather Information API Tool', True), ('Final Response Formatter', True), ('Campus Announcement Generator', True)]

FINAL:
 {
  "Event Plan": "The event is scheduled from 09:00 to 17:00, including a welcome session, keynote lecture, student interaction, lunch, transport coordination, and departure planning. The exact date and venue are yet to be confirmed. The schedule includes buffer times to handle potential delays.",
  "Schedule": "The tentative timetable is as follows: 09:00-10:20 Welcome Session, 10:20-11:40 Keynote Lecture, 11:40-13:00 Student Interaction, 13:00-14:20 Lunch, 14:20-15:40 Transport Coordination, 15:40-17:00 Departure Plan. Exact timings and activities may be adjusted based on final confirmations.",
  "Resources": "Essential resources include seating for 150 attendees,

In [ ]:
# TEST 3: Announcement-only (placeholders + organiser checklist, no invented links).
test3 = ("Prepare an announcement for an upcoming campus coding competition "
         "for students. Include participation instructions and a checklist "
         "of information that organisers still need to provide.")
print("TEST 3 INPUT:", test3)
if NEXUSAI_OK:
    TOOL_LOG.clear()
    r3 = run_campus_assistant(test3)
    print("\nTOOLS:", [(e["tool"], e["ok"]) for e in r3["tools_used"]])
    print("\nFINAL:\n", r3["final"][:4000])
    print("\nCHECKS: [TBD] for date/venue/link? organiser checklist "
          "included? zero invented registration URLs?")
else:
    print("NOT RUN: Nexus key missing - safe refusal recorded.")


## SECTION 12 — Failure Testing (2 simulated scenarios)

- **A: Incomplete request** — real validator path on `"Organise a college event."`: must ask follow-ups or plan provisionally with labelled assumptions, never inventing numbers.
- **B: Weather API failure** — SIMULATED via `unittest.mock.patch` forcing `requests.get` to raise `Timeout` (clearly labelled SIMULATED). Proves Tool 5 degrades to `{"status":"unavailable"}` with no fabricated forecast, while the normal Tool 5 path (Section 6 demo) uses real Open-Meteo data.


In [23]:
# FAILURE A: incomplete request (uses real code, no crew needed).
import json as _json
fail_a = "Organise a college event."
print("FAILURE A INPUT:", fail_a)
_val = validate_request(fail_a)
print("Validator output:", _json.dumps(_val, indent=2))
_an = _json.loads(RequirementsAnalyzerTool()._run(fail_a))
print("Analyzer output:", _json.dumps(_an, indent=2))
print("Missing info reported:", _an.get("missing_information"))
print("Assumptions labelled:", _an.get("assumptions"))
print("Outcome: workflow asks the follow-ups above OR proceeds with the "
      "labelled assumptions - no dates, venues or budgets invented. "
      "Continued safely: YES.")


FAILURE A INPUT: Organise a college event.
Validator output: {
  "status": "ok",
  "missing": [
    "headcount",
    "venue",
    "date"
  ],
  "followups": [
    "Please confirm: headcount?",
    "Please confirm: venue?",
    "Please confirm: date?"
  ]
}
Analyzer output: {
  "facts_user_provided": {
    "event_type": "general campus event"
  },
  "assumptions": {
    "duration_default": "one-day (09:00-17:00) assumed for planning",
    "audience_default": "150 assumed for planning"
  },
  "missing_information": [
    "audience size",
    "date",
    "venue",
    "duration"
  ],
  "clarifying_questions": [
    "Please confirm: audience size?",
    "Please confirm: date?",
    "Please confirm: venue?",
    "Please confirm: duration?"
  ]
}
Missing info reported: ['audience size', 'date', 'venue', 'duration']
Assumptions labelled: {'duration_default': 'one-day (09:00-17:00) assumed for planning', 'audience_default': '150 assumed for planning'}
Outcome: workflow asks the follow-ups above

In [24]:
# FAILURE B: SIMULATED weather API timeout (deterministic, via mock).
# The real-data path in Tool 5 (Sec 6) is untouched; here we force
# requests.get to raise Timeout so we can verify the tool degrades
# to {"status": "unavailable"} without fabricating a forecast.
from unittest.mock import patch
import requests as _rq
print("FAILURE B: SIMULATED timeout test (mocked requests.get).")
with patch.object(_rq, "get", side_effect=_rq.exceptions.Timeout("simulated timeout")):
    tool_msg = WeatherInfoTool()._run(13.2146, 74.9967, 3)
print("Tool on simulated timeout:", tool_msg)
parsed = json.loads(tool_msg)
print("Status field:", parsed.get("status"))
assert parsed.get("status") == "unavailable", "must degrade to unavailable"
assert "temperature" not in parsed or parsed.get("temperature") is None,     "must not fabricate a forecast on failure"
print("Outcome: Timeout caught, 'unavailable' returned, crew continues "
      "with indoor contingency. No forecast fabricated. Continued: YES.")


FAILURE B: SIMULATED timeout test (mocked requests.get).
Tool on simulated timeout: {"status": "unavailable", "reason": "timeout", "guidance": "Proceed with indoor contingency."}
Status field: unavailable
Outcome: Timeout caught, 'unavailable' returned, crew continues with indoor contingency. No forecast fabricated. Continued: YES.


## SECTION 13 — Debug Log

Record only **genuine or intentionally reproduced** issues. Rows 1-2 are pre-labelled REPRODUCED-TEST; row 3 is a GENUINE Colab failure with real evidence (we trigger them on purpose in Section 12); the third row is a template you fill from your own Colab run. Do not invent errors that never happened.

| # | Problem | Error / Observed behaviour | Root cause | Solution implemented | Remaining limitations |
|---|---|---|---|---|---|
| 1 | Weather API failure path (REPRODUCED-TEST, Sec 12-B) | Mocked `requests.get` → `Timeout` (simulated) | Forced timeout via `unittest.mock.patch` | Tool returns `{"status":"unavailable",...}` (asserted, no forecast); crew uses indoor contingency | Real outages still need retry/backoff |
| 2 | Incomplete request handling (REPRODUCED-TEST, Sec 12-A) | `"Organise a college event."` → missing headcount/date/venue | User gave no specifics | Validator + analyzer list follow-ups; provisional plan labelled assumptions | Cannot guess real date/venue |
| 3 | Sync `crew.kickoff()` inside Colab (GENUINE, Sec 10-11) | `RuntimeError: Agent execution was invoked synchronously from within a running event loop` | Colab/Jupyter already runs an event loop; CrewAI forbids sync kickoff inside it | Runner now prefers `crew.kickoff_async()` in a fresh thread+loop (`_crew_run` helper); verified pattern from `03_multi_agent` notebook | Re-run Sec 10-11 in Colab to confirm final responses generate |


## SECTION 14 — Results and Observations

Evidence below is from the **executed Colab run, 15 Oct 2026** (CrewAI 1.15.28, Nexus key length 25, `Nexus test reply: NEXUS-OK`):

| Check | Evidence location | Result (write what you see) |
|---|---|---|
| Agents defined + ready | Sec 7 output | YES — `4 agents ready: Campus Event Planning Coordinator \| Campus Logistics and Budget Manager \| Event Risk and Contingency Analyst \| Campus Communications Specialist` |
| Task context chaining defined | Sec 8 `build_tasks` (`context=[t1]`, `[t1,t2]`, `[t1,t2,t3]`) + Sec 10 task plan (T1→T4 printed) | YES by code; downstream sections reference upstream facts once crew runs |
| All 8 tools defined + demoed | Sec 6a/6b demos + `TOOL_LOG` | YES — Tool 1 (`ai fest`, 300, one-day, NITTE), Tool 2 (8h, 7 slots, 69 min), Tool 3 (budget totals), Tool 4 (checklist), Tool 5 (live, see below), Tool 6 (Nexus announcement, 120-180 words + `[TBD]`), Tool 7 (risks scored), Tool 8 (sections merged) |
| External API returned real data | Sec 5 + Tool 5 demo (HTTP 200) | YES — 15 Oct 2026 19:15 IST, NITTE resolved lat 13.181019/lon 74.98633, now 26.4°C (feels 32.3), RH 95%, wind 7.5 km/h, Overcast (WMO 3); daily rain prob 82%/86% + Thunderstorm 17th (94%) — indoor backup required |
| Crew execution (Sec 10-11) | Sec 10/11 outputs | BLOCKED at run time by event-loop bug (see Debug Log #3): `RuntimeError: ... running event loop`; task plan printed, `TOOLS: []`, safe-failure message returned. Fix implemented (`_crew_run` → `kickoff_async`); **re-run Sec 10-11 in Colab to capture FINAL blocks** |
| Missing info handled safely | Failure A output | YES — `"Organise a college event."` → validator `missing: [headcount, venue, date]` + follow-ups; analyzer `missing_information: [audience size, date, venue, duration]`, assumptions labelled (one-day 09:00-17:00, 150 attendees); no facts invented |
| Simulated failures graceful | Sec 12 A/B outputs | YES — A continues with assumptions; B (mocked `Timeout`) → `{"status":"unavailable"}` asserted, no forecast fabricated |
| Improvements | — | Re-run crew after `_crew_run` fix; add retry/backoff for weather; venue-capacity table; date parser for Task 1; token-usage log |


## SECTION 15 — Conclusion

**How it works:** one natural-language request flows through four specialised CrewAI agents in sequence. The Coordinator structures the ask (Tools 1-2), Logistics costs and resources it (Tools 3-4), the Risk analyst grounds it in live Open-Meteo weather and scored mitigations (Tools 5, 7), and Communications renders the Nexus-drafted announcement plus a deterministically merged final response (Tools 6, 8).

**CrewAI coordination:** `Crew(process=Process.sequential)` with explicit `context=[...]` passes Task 1's facts into Tasks 2-4, so downstream agents reuse rather than re-invent information; each task names its tools so invocation is genuine.

**Tools improve reliability:** money, timetables and severity use Python rules (auditable, repeatable); the LLM is reserved for planning prose and announcements — always via Nexus, never another provider.

**External APIs:** Open-Meteo supplies keyless, verifiable weather inside its 16-day horizon; failures degrade to labelled `unavailable` + indoor contingency instead of fabricated forecasts.

**Limitations:** needs `NEXUSAI_API_KEY` for crew/announcement steps; default NITTE coordinates unless the user supplies lat/lon; cost figures are estimates; dates beyond 16 days cannot use real forecasts.

**Future work:** venue database + capacity checks, multi-day scheduling, e-mail/calendar actions with explicit human approval gates, caching + retries, and a Streamlit/Colab-form front end.
